# Capitolo 17 – Algoritmi ricorsivi: somma, Merge Sort, ricerca binaria, sottoarray massimo (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi_java.ipynb)

Gli algoritmi con pseudocodice del capitolo, eseguiti sugli esempi delle dispense: la **somma di un array** (iterativa, per dimezzamento e per decremento di uno), **Merge Sort** con `Merge`, la **ricerca binaria ricorsiva** e il problema del **sottoarray massimo** (forza bruta, divide et impera, algoritmo di Kadane). Le stampe riproducono le tracce e i numeri del capitolo (con gli indici da 0).

Il capitolo completo, con pseudocodice, dimostrazioni ed esempi, è [sul sito](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/17-algoritmi-ricorsivi/).

Ogni programma è scritto in un file `.java` con la cella `%%writefile` e poi compilato ed eseguito con `javac` e `java`: si eseguono le celle in ordine, dall'alto in basso (la prima cella verifica o installa il JDK). Nessuna libreria esterna.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `RecursiveSum` → `recursiveSum` (camelCase).

In [1]:
# Verifica (ed eventualmente installa) il JDK
!java -version || apt-get -qq install -y default-jdk-headless

openjdk version "25.0.4.1" 2026-08-18
OpenJDK Runtime Environment (build 25.0.4.1+1-1-26.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 25.0.4.1+1-1-26.04.4-Ubuntu, mixed mode, sharing)


## 1. Somma di un array: SumIterative, RecursiveSum, RecursiveSumLinear

- **SumIterative**$(a,n)$: un ciclo che accumula le somme parziali. $S(n)=3n+4$, memoria extra $\Theta(1)$.
- **RecursiveSum**$(a,l,r)$: divide et impera sulle due metà (caso base $l=r$). $2n-1$ chiamate, $S(n)=7n-5$, profondità $\lceil\log_2 n\rceil+1$.
- **RecursiveSumLinear**$(a,n)$: decremento di uno: $S(n)=3n-1$, profondità $n$.

*Input*: un array di $n\ge1$ interi (per `RecursiveSum`, due indici $l\le r$). *Output*: la somma degli elementi. I contatori `steps` (righe di pseudocodice eseguite), `calls` e `maxDepth` (chiamate contemporaneamente attive) misurano le grandezze analizzate nel capitolo; in `RecursiveSum` e `RecursiveSumLinear` la `return` è unica per poter aggiornare la profondità.

Sull'array dell'esempio $a=[9,8,12,3,5,14,6]$ ($n=7$) la somma è $57$; `RecursiveSum` fa $13$ chiamate e $44$ passi, con profondità $4$. Poi si esegue la traccia della tabella del capitolo su $a=[9,8,12,3,5]$: la colonna *stack* riporta le chiamate attive.

In [2]:
%%writefile Sums.java
import java.util.*;

public class Sums {
    static long steps = 0;   // passi eseguiti (righe di pseudocodice)
    static int calls = 0;    // chiamate eseguite
    static int depth = 0;    // chiamate attive in questo momento
    static int maxDepth = 0; // profondità massima della ricorsione

    static void reset() {
        steps = 0;
        calls = 0;
        depth = 0;
        maxDepth = 0;
    }

    // Somma iterativa di a[0..n-1]
    static int sumIterative(int[] a, int n) {
        int s = 0;
        int i = 0;
        steps += 2;                 // righe 1-2: s := 0, i := 1
        while (i < n) {
            steps += 3;             // test vero + due righe del corpo
            s = s + a[i];
            i = i + 1;
        }
        steps += 2;                 // test falso + return
        return s;
    }

    // Somma di a[l..r] (indici da 0), l <= r, per divide et impera
    static int recursiveSum(int[] a, int l, int r) {
        calls++;
        depth++;
        if (depth > maxDepth) maxDepth = depth;
        steps++;                    // riga 1: if l = r
        int result;
        if (l == r) {
            steps++;                // riga 2: return a[l] (caso base)
            result = a[l];
        } else {
            steps += 4;             // righe 3-6: m, due chiamate, return
            int m = l + (r - l) / 2;                // divide
            int s1 = recursiveSum(a, l, m);         // conquer (metà sinistra)
            int s2 = recursiveSum(a, m + 1, r);     // conquer (metà destra)
            result = s1 + s2;                       // combine
        }
        depth--;
        return result;
    }

    // Somma di a[0..n-1] per decremento di uno, n >= 1
    static int recursiveSumLinear(int[] a, int n) {
        calls++;
        depth++;
        if (depth > maxDepth) maxDepth = depth;
        steps++;                    // riga 1: if n = 1
        int result;
        if (n == 1) {
            steps++;                // riga 2: return a[1]
            result = a[0];
        } else {
            steps += 2;             // righe 3-4: chiamata e return
            int s = recursiveSumLinear(a, n - 1);
            result = s + a[n - 1];
        }
        depth--;
        return result;
    }

    // ---- traccia di RecursiveSum ----
    static int row = 0;
    static int maxStack = 0;

    static void emit(ArrayList<String> stack, String action) {
        row++;
        if (stack.size() > maxStack) maxStack = stack.size();
        System.out.printf("%2d | %s | %s%n", row, String.join(",", stack), action);
    }

    static String p(int l, int r) { return "(" + l + "," + r + ")"; }

    static int recursiveSumTrace(int[] a, int l, int r, ArrayList<String> stack) {
        stack.add(p(l, r));
        if (l == r) {
            emit(stack, "l=r => return a[" + l + "]=" + a[l]);
            stack.remove(stack.size() - 1);
            return a[l];
        }
        int m = l + (r - l) / 2;
        emit(stack, "l!=r, m=" + l + "+(" + r + "-" + l + ")/2=" + m + "; call " + p(l, m));
        int s1 = recursiveSumTrace(a, l, m, stack);
        emit(stack, "s1=" + s1 + "; call " + p(m + 1, r));
        int s2 = recursiveSumTrace(a, m + 1, r, stack);
        emit(stack, "s2=" + s2 + "; return " + s1 + "+" + s2 + "=" + (s1 + s2));
        stack.remove(stack.size() - 1);
        return s1 + s2;
    }

    public static void main(String[] args) {
        int[] a = {9, 8, 12, 3, 5, 14, 6};
        int n = 7;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        reset();
        int s = sumIterative(a, n);
        System.out.println("SumIterative       = " + s + "   steps=" + steps + " (3n+4=" + (3 * n + 4) + ")");
        reset();
        s = recursiveSum(a, 0, n - 1);
        System.out.println("RecursiveSum       = " + s + "   calls=" + calls + " (2n-1=" + (2 * n - 1) + ")  steps=" + steps + " (7n-5=" + (7 * n - 5) + ")  maxDepth=" + maxDepth);
        reset();
        s = recursiveSumLinear(a, n);
        System.out.println("RecursiveSumLinear = " + s + "   calls=" + calls + "  steps=" + steps + " (3n-1=" + (3 * n - 1) + ")  maxDepth=" + maxDepth);

        int[] b = {9, 8, 12, 3, 5};
        System.out.println();
        System.out.println("a = " + Arrays.toString(b) + "   RecursiveSum(a, 0, 4)");
        System.out.println(" # | stack | azione");
        int res = recursiveSumTrace(b, 0, 4, new ArrayList<>());
        System.out.println("risultato " + res + ", righe " + row + ", massimo numero di chiamate attive " + maxStack);
    }
}

Writing Sums.java


In [3]:
!javac Sums.java && java Sums

a = [9, 8, 12, 3, 5, 14, 6]   n = 7
SumIterative       = 57   steps=25 (3n+4=25)
RecursiveSum       = 57   calls=13 (2n-1=13)  steps=44 (7n-5=44)  maxDepth=4
RecursiveSumLinear = 57   calls=7  steps=20 (3n-1=20)  maxDepth=7

a = [9, 8, 12, 3, 5]   RecursiveSum(a, 0, 4)
 # | stack | azione
 1 | (0,4) | l!=r, m=0+(4-0)/2=2; call (0,2)
 2 | (0,4),(0,2) | l!=r, m=0+(2-0)/2=1; call (0,1)
 3 | (0,4),(0,2),(0,1) | l!=r, m=0+(1-0)/2=0; call (0,0)
 4 | (0,4),(0,2),(0,1),(0,0) | l=r => return a[0]=9
 5 | (0,4),(0,2),(0,1) | s1=9; call (1,1)
 6 | (0,4),(0,2),(0,1),(1,1) | l=r => return a[1]=8
 7 | (0,4),(0,2),(0,1) | s2=8; return 9+8=17
 8 | (0,4),(0,2) | s1=17; call (2,2)
 9 | (0,4),(0,2),(2,2) | l=r => return a[2]=12
10 | (0,4),(0,2) | s2=12; return 17+12=29
11 | (0,4) | s1=29; call (3,4)
12 | (0,4),(3,4) | l!=r, m=3+(4-3)/2=3; call (3,3)
13 | (0,4),(3,4),(3,3) | l=r => return a[3]=3
14 | (0,4),(3,4) | s1=3; call (4,4)
15 | (0,4),(3,4),(4,4) | l=r => return a[4]=5
16 | (0,4),(3,4) | s2=5; retur

**Prova tu.** Cambia l'array `a` (anche la lunghezza) e confronta passi, chiamate e profondità: solo `RecursiveSumLinear` ha profondità $n$.

In [4]:
%%writefile ProvaSums.java
import java.util.*;

public class ProvaSums {
    public static void main(String[] args) {
        // Cambia l'array: confronta passi, chiamate e profondità dei tre algoritmi.
        int[] a = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16};
        int n = a.length;
        Sums.reset(); int s = Sums.sumIterative(a, n);
        System.out.println(s + " " + Sums.steps);
        Sums.reset(); s = Sums.recursiveSum(a, 0, n - 1);
        System.out.println(s + " " + Sums.steps + " " + Sums.calls + " " + Sums.maxDepth);
        Sums.reset(); s = Sums.recursiveSumLinear(a, n);
        System.out.println(s + " " + Sums.steps + " " + Sums.calls + " " + Sums.maxDepth);
    }
}

Writing ProvaSums.java


In [5]:
!javac ProvaSums.java && java ProvaSums

136 52
136 107 31 5
136 47 16 16


## 2. Merge Sort e Merge

- **MergeSort**$(a,l,r)$ — *Input*: un array e due indici $l\le r$. *Output*: il sotto-array $a[l..r]$ ordinato in modo non decrescente. Si chiama `MergeSort(a, 1, n)` nello pseudocodice, `merge_sort(a, 0, n-1)` nel codice.
- **Merge**$(a,l,m,r)$ — *Input*: un array e indici $l\le m<r$ tali che $a[l..m]$ e $a[m+1..r]$ siano ordinati. *Output*: $a[l..r]$ ordinato. Copia le due metà in $L$ e $R$ e riversa in $a$, passo dopo passo, il minore fra i primi due elementi non ancora copiati.

Il contatore `comparisons` conta i confronti fra elementi eseguiti da `Merge`. Sull'array dell'esempio $a=[9,12,3,16,19,11,12]$ si stampano le chiamate di `Merge` nell'ordine di esecuzione (con l'array dopo ciascuna, tabella del capitolo; gli indici sono da 0: aggiungendo 1 si ritrovano quelli del capitolo), poi la traccia di `Merge(a, 0, 1, 3)` su $a=[5,8,2,7]$ (esempio dell'implementazione in C++).

In [6]:
%%writefile MergeSort.java
import java.util.*;

public class MergeSort {
    static long comparisons = 0;   // confronti fra elementi eseguiti da Merge
    static int msCalls = 0;        // chiamate di MergeSort

    // merge(a, l, m, r): fonde a[l..m] e a[m+1..r], entrambi ordinati
    static void merge(int[] a, int l, int m, int r) {
        int n1 = m - l + 1;             // dimensione della parte sinistra
        int n2 = r - m;                 // dimensione della parte destra
        int[] L = new int[n1];          // array ausiliari
        int[] R = new int[n2];
        for (int i = 0; i < n1; i++) {
            L[i] = a[l + i];
        }
        for (int j = 0; j < n2; j++) {
            R[j] = a[m + 1 + j];
        }
        int i = 0, j = 0, k = l;
        while (i < n1 && j < n2) {      // fonde L e R in a[l..r]
            comparisons++;
            if (L[i] <= R[j]) {
                a[k] = L[i];
                i++;
            } else {
                a[k] = R[j];
                j++;
            }
            k++;
        }
        while (i < n1) {                // copia il resto di L
            a[k] = L[i];
            i++;
            k++;
        }
        while (j < n2) {                // copia il resto di R
            a[k] = R[j];
            j++;
            k++;
        }
    }

    // mergeSort(a, l, r): ordina a[l..r]
    static void mergeSort(int[] a, int l, int r) {
        msCalls++;
        if (l < r) {
            int m = l + (r - l) / 2;    // divide
            mergeSort(a, l, m);         // conquer (metà sinistra)
            mergeSort(a, m + 1, r);     // conquer (metà destra)
            merge(a, l, m, r);          // combine
        }
    }

    // ---- varianti che stampano la traccia ----
    static void mergeSortLog(int[] a, int l, int r) {
        if (l < r) {
            int m = l + (r - l) / 2;
            mergeSortLog(a, l, m);
            mergeSortLog(a, m + 1, r);
            String left = Arrays.toString(Arrays.copyOfRange(a, l, m + 1));
            String right = Arrays.toString(Arrays.copyOfRange(a, m + 1, r + 1));
            merge(a, l, m, r);
            System.out.println("Merge(a," + l + "," + m + "," + r + ")  " + left + " e " + right + "  ->  " + Arrays.toString(a));
        }
    }

    static void mergeTrace(int[] a, int l, int m, int r) {
        int n1 = m - l + 1, n2 = r - m;
        int[] L = Arrays.copyOfRange(a, l, m + 1), R = Arrays.copyOfRange(a, m + 1, r + 1);
        System.out.println("Merge(a, " + l + ", " + m + ", " + r + ") su a = " + Arrays.toString(a));
        System.out.println("n1=" + n1 + ", n2=" + n2 + ", L=" + Arrays.toString(L) + ", R=" + Arrays.toString(R));
        int i = 0, j = 0, k = l, it = 0;
        while (i < n1 && j < n2) {
            it++;
            String head;
            if (L[i] <= R[j]) {
                head = String.format("%2d. ciclo 1: L[%d]=%d <= R[%d]=%d? Sì => a[%d] = L[%d] = %d", it, i, L[i], j, R[j], k, i, L[i]);
                a[k] = L[i];
                i++;
            } else {
                head = String.format("%2d. ciclo 1: L[%d]=%d <= R[%d]=%d? No => a[%d] = R[%d] = %d", it, i, L[i], j, R[j], k, j, R[j]);
                a[k] = R[j];
                j++;
            }
            k++;
            System.out.println(head + ", i=" + i + ", j=" + j + ", k=" + k + "; a=" + Arrays.toString(a));
        }
        while (i < n1) {
            a[k] = L[i];
            System.out.println("    ciclo 2 (resto di L): a[" + k + "] = L[" + i + "] = " + L[i] + "; a=" + Arrays.toString(a));
            i++;
            k++;
        }
        while (j < n2) {
            a[k] = R[j];
            System.out.println("    ciclo 3 (resto di R): a[" + k + "] = R[" + j + "] = " + R[j] + "; a=" + Arrays.toString(a));
            j++;
            k++;
        }
        System.out.println("risultato a = " + Arrays.toString(a) + ", confronti = " + it);
    }

    public static void main(String[] args) {
        int[] a = {9, 12, 3, 16, 19, 11, 12};
        int n = 7;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        System.out.println("Chiamate di Merge in ordine di esecuzione:");
        mergeSortLog(a.clone(), 0, n - 1);
        comparisons = 0;
        msCalls = 0;
        mergeSort(a, 0, n - 1);
        System.out.println("ordinato = " + Arrays.toString(a) + "   chiamate di MergeSort=" + msCalls + " (2n-1=" + (2 * n - 1) + ")   confronti=" + comparisons);
        System.out.println();
        int[] b = {5, 8, 2, 7};
        mergeTrace(b, 0, 1, 3);
    }
}

Writing MergeSort.java


In [7]:
!javac MergeSort.java && java MergeSort

a = [9, 12, 3, 16, 19, 11, 12]   n = 7
Chiamate di Merge in ordine di esecuzione:
Merge(a,0,0,1)  [9] e [12]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,2,2,3)  [3] e [16]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,0,1,3)  [9, 12] e [3, 16]  ->  [3, 9, 12, 16, 19, 11, 12]
Merge(a,4,4,5)  [19] e [11]  ->  [3, 9, 12, 16, 11, 19, 12]
Merge(a,4,5,6)  [11, 19] e [12]  ->  [3, 9, 12, 16, 11, 12, 19]
Merge(a,0,3,6)  [3, 9, 12, 16] e [11, 12, 19]  ->  [3, 9, 11, 12, 12, 16, 19]
ordinato = [3, 9, 11, 12, 12, 16, 19]   chiamate di MergeSort=13 (2n-1=13)   confronti=14

Merge(a, 0, 1, 3) su a = [5, 8, 2, 7]
n1=2, n2=2, L=[5, 8], R=[2, 7]
 1. ciclo 1: L[0]=5 <= R[0]=2? No => a[0] = R[0] = 2, i=0, j=1, k=1; a=[2, 8, 2, 7]
 2. ciclo 1: L[0]=5 <= R[1]=7? Sì => a[1] = L[0] = 5, i=1, j=1, k=2; a=[2, 5, 2, 7]
 3. ciclo 1: L[1]=8 <= R[1]=7? No => a[2] = R[1] = 7, i=1, j=2, k=3; a=[2, 5, 7, 7]
    ciclo 2 (resto di L): a[3] = L[1] = 8; a=[2, 5, 7, 8]
risultato a = [2, 5, 7, 8], confronti = 3


**Prova tu.** Ordina un altro array: il numero di confronti di `Merge` è al massimo $n\lceil\log_2 n\rceil$ e le chiamate di `MergeSort` sono sempre $2n-1$.

In [8]:
%%writefile ProvaMergeSort.java
import java.util.*;

public class ProvaMergeSort {
    public static void main(String[] args) {
        // Ordina un altro array e conta i confronti (al massimo n*ceil(log2 n)).
        int[] b = {38, 27, 43, 3, 9, 82, 10, 5};
        MergeSort.comparisons = 0;
        MergeSort.msCalls = 0;
        MergeSort.mergeSort(b, 0, b.length - 1);
        System.out.println(Arrays.toString(b) + " " + MergeSort.comparisons + " " + MergeSort.msCalls);
    }
}

Writing ProvaMergeSort.java


In [9]:
!javac ProvaMergeSort.java && java ProvaMergeSort

[3, 5, 9, 10, 27, 38, 43, 82] 17 15


## 3. RecursiveBinarySearch

**RecursiveBinarySearch**$(a,left,right,m)$ — *Input*: un array $a$ di naturali ordinato in modo non decrescente, due indici $left$ e $right$ e un valore $m$. *Output*: un indice $i\in\{left,\dots,right\}$ con $a[i]=m$, oppure $-1$ se $m$ non è in $a[left..right]$. Ogni chiamata fa **al più una** chiamata ricorsiva sulla metà (circa) dell'intervallo: l'albero delle chiamate è un cammino, con al più $\lfloor\log_2 n\rfloor+2$ chiamate. Per cercare in tutto l'array si chiama `RecursiveBinarySearch(a, 0, n-1, m)`.

Sullo stesso array $a=[3,9,11,12,12,16,19]$ del capitolo si cercano $m=12,10,3,19$. Per ogni ricerca la tabella ha una riga per chiamata ($left$, $right$, $mid$, $a[mid]$, azione), come nel capitolo ma con gli indici da 0: aggiungendo 1 si ritrovano quelli del capitolo (per esempio $m=19$ è trovato all'indice 6, cioè alla posizione 7).

In [10]:
%%writefile BinarySearch.java
import java.util.*;

public class BinarySearch {
    static int rbsCalls = 0;   // chiamate eseguite

    // Indice di m in a[left..right] (array ordinato), oppure -1
    static int recursiveBinarySearch(int[] a, int left, int right, int m) {
        rbsCalls++;
        if (left > right) {
            return -1;                  // intervallo vuoto: non trovato
        }
        int mid = left + (right - left) / 2;
        if (a[mid] == m) {
            return mid;                 // trovato
        }
        if (a[mid] < m) {
            return recursiveBinarySearch(a, mid + 1, right, m);    // metà destra
        }
        return recursiveBinarySearch(a, left, mid - 1, m);         // metà sinistra
    }

    // Come sopra, ma stampa una riga per chiamata (k = numero della chiamata)
    static int rbsTrace(int[] a, int left, int right, int m, int k) {
        if (left > right) {
            System.out.printf("%4d %4d %5d %3s %6s left > right => non trovato, return -1%n", k, left, right, "-", "-");
            return -1;
        }
        int mid = left + (right - left) / 2;
        System.out.printf("%4d %4d %5d %3d %6d ", k, left, right, mid, a[mid]);
        if (a[mid] == m) {
            System.out.println("a[mid] = m => trovato, return " + mid);
            return mid;
        }
        if (a[mid] < m) {
            System.out.println("a[mid] < m => chiamata su left=" + (mid + 1) + ", right=" + right);
            return rbsTrace(a, mid + 1, right, m, k + 1);
        }
        System.out.println("a[mid] > m => chiamata su left=" + left + ", right=" + (mid - 1));
        return rbsTrace(a, left, mid - 1, m, k + 1);
    }

    public static void main(String[] args) {
        int[] a = {3, 9, 11, 12, 12, 16, 19};
        int n = 7;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        int[] ms = {12, 10, 3, 19};
        for (int m : ms) {
            System.out.println();
            System.out.println("Ricerca di m = " + m);
            System.out.printf("%4s %4s %5s %3s %6s azione%n", "call", "left", "right", "mid", "a[mid]");
            rbsTrace(a, 0, n - 1, m, 1);
        }
        System.out.println();
        for (int m : ms) {
            rbsCalls = 0;
            int r = recursiveBinarySearch(a, 0, n - 1, m);
            System.out.printf("m=%-3d -> %2d (%2d con indici da 1)  chiamate=%d%n", m, r, r >= 0 ? r + 1 : -1, rbsCalls);
        }
    }
}

Writing BinarySearch.java


In [11]:
!javac BinarySearch.java && java BinarySearch

a = [3, 9, 11, 12, 12, 16, 19]   n = 7

Ricerca di m = 12
call left right mid a[mid] azione
   1    0     6   3     12 a[mid] = m => trovato, return 3

Ricerca di m = 10
call left right mid a[mid] azione
   1    0     6   3     12 a[mid] > m => chiamata su left=0, right=2
   2    0     2   1      9 a[mid] < m => chiamata su left=2, right=2
   3    2     2   2     11 a[mid] > m => chiamata su left=2, right=1
   4    2     1   -      - left > right => non trovato, return -1

Ricerca di m = 3
call left right mid a[mid] azione
   1    0     6   3     12 a[mid] > m => chiamata su left=0, right=2
   2    0     2   1      9 a[mid] > m => chiamata su left=0, right=0
   3    0     0   0      3 a[mid] = m => trovato, return 0

Ricerca di m = 19
call left right mid a[mid] azione
   1    0     6   3     12 a[mid] < m => chiamata su left=4, right=6
   2    4     6   5     16 a[mid] < m => chiamata su left=6, right=6
   3    6     6   6     19 a[mid] = m => trovato, return 6

m=12  ->  3 ( 4 con ind

**Prova tu.** Cerca un altro valore (presente o assente) e verifica che le chiamate siano al più $\lfloor\log_2 n\rfloor+2$.

In [12]:
%%writefile ProvaBinarySearch.java
import java.util.*;

public class ProvaBinarySearch {
    public static void main(String[] args) {
        // Cerca un altro valore, anche assente: le chiamate sono al più floor(log2 n) + 2.
        int[] a = {3, 9, 11, 12, 12, 16, 19};
        BinarySearch.rbsCalls = 0;
        System.out.println(BinarySearch.recursiveBinarySearch(a, 0, a.length - 1, 16) + " " + BinarySearch.rbsCalls);
        BinarySearch.rbsCalls = 0;
        System.out.println(BinarySearch.recursiveBinarySearch(a, 0, a.length - 1, 100) + " " + BinarySearch.rbsCalls);
    }
}

Writing ProvaBinarySearch.java


In [13]:
!javac ProvaBinarySearch.java && java ProvaBinarySearch

5 2
-1 4


## 4. BruteForceMaximumSubarray

**BruteForceMaximumSubarray**$(a,n)$ — *Input*: un array $a[1..n]$ di $n$ interi. *Output*: gli indici $bl\le br$ di un sottoarray di somma massima e la somma $best$. Esamina tutti gli $n(n+1)/2$ sottoarray $a[l..r]$ (non vuoti, di elementi consecutivi), calcolando le somme in modo incrementale: $\Theta(n^2)$.

Sull'esempio $a=[-2,5,-1,3]$ del capitolo la tabella ha una riga per sottoarray esaminato ($l$, $r$, somma, azione; in ogni riga che non migliora si indica $best$ corrente). Gli indici sono da 0: il capitolo trova $(2,4,7)$, cioè $a[2..4]=[5,-1,3]$ di somma $7$, qui $(1,3,7)$.

In [14]:
%%writefile MaxSubBrute.java
import java.util.*;

public class MaxSubBrute {
    // Restituisce {bl, br, best}: un sottoarray di somma massima di a[0..n-1]
    static int[] bruteForceMaximumSubarray(int[] a, int n) {
        int best = a[0];
        int bl = 0;
        int br = 0;
        int l = 0;
        while (l < n) {
            int sum = 0;
            int r = l;
            while (r < n) {
                sum = sum + a[r];             // somma di a[l..r]
                if (sum > best) {
                    best = sum;
                    bl = l;
                    br = r;
                }
                r = r + 1;
            }
            l = l + 1;
        }
        return new int[]{bl, br, best};
    }

    // Come sopra, ma stampa una riga per ogni sottoarray esaminato
    static int[] bruteForceTrace(int[] a, int n) {
        int best = a[0];
        int bl = 0;
        int br = 0;
        for (int l = 0; l < n; l++) {
            int sum = 0;
            for (int r = l; r < n; r++) {
                sum = sum + a[r];
                if (sum > best) {
                    best = sum;
                    bl = l;
                    br = r;
                    System.out.printf("%3d %3d %5d  sum > best => best=%d, bl=%d, br=%d%n", l, r, sum, best, bl, br);
                } else {
                    System.out.printf("%3d %3d %5d  sum <= best=%d%n", l, r, sum, best);
                }
            }
        }
        return new int[]{bl, br, best};
    }

    public static void main(String[] args) {
        int[] a = {-2, 5, -1, 3};
        int n = 4;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n + "   sottoarray n(n+1)/2 = " + n * (n + 1) / 2);
        System.out.printf("%3s %3s %5s  azione%n", "l", "r", "sum");
        int[] t = bruteForceTrace(a, n);
        System.out.printf("risultato (bl, br, best) = (%d, %d, %d)   (%d, %d, %d con indici da 1)%n", t[0], t[1], t[2], t[0] + 1, t[1] + 1, t[2]);
        int[] u = bruteForceMaximumSubarray(a, n);
        System.out.printf("BruteForceMaximumSubarray(a, n) = (%d, %d, %d)%n", u[0], u[1], u[2]);
    }
}

Writing MaxSubBrute.java


In [15]:
!javac MaxSubBrute.java && java MaxSubBrute

a = [-2, 5, -1, 3]   n = 4   sottoarray n(n+1)/2 = 10
  l   r   sum  azione
  0   0    -2  sum <= best=-2
  0   1     3  sum > best => best=3, bl=0, br=1
  0   2     2  sum <= best=3
  0   3     5  sum > best => best=5, bl=0, br=3
  1   1     5  sum <= best=5
  1   2     4  sum <= best=5
  1   3     7  sum > best => best=7, bl=1, br=3
  2   2    -1  sum <= best=7
  2   3     2  sum <= best=7
  3   3     3  sum <= best=7
risultato (bl, br, best) = (1, 3, 7)   (2, 4, 7 con indici da 1)
BruteForceMaximumSubarray(a, n) = (1, 3, 7)


**Prova tu.** Cambia l'array (per esempio con soli numeri negativi): il sottoarray massimo è il singolo elemento maggiore.

In [16]:
%%writefile ProvaMaxSubBrute.java
import java.util.*;

public class ProvaMaxSubBrute {
    public static void main(String[] args) {
        // Prova con un array di soli numeri negativi: il sottoarray massimo è il singolo elemento maggiore.
        int[] b = {-3, -1, -2, -5};
        int[] t = MaxSubBrute.bruteForceMaximumSubarray(b, b.length);
        System.out.println("(" + t[0] + ", " + t[1] + ", " + t[2] + ")");
    }
}

Writing ProvaMaxSubBrute.java


In [17]:
!javac ProvaMaxSubBrute.java && java ProvaMaxSubBrute

(1, 1, -1)


## 5. FindMaxCrossingSubarray e FindMaximumSubarray (divide et impera)

- **FindMaxCrossingSubarray**$(a,l,mid,r)$ — *Input*: un array e tre indici $l\le mid<r$. *Output*: gli indici $maxleft\le mid<maxright$ di un sottoarray di somma massima fra quelli che attraversano il punto medio, e la sua somma. Scandisce la metà sinistra da $mid$ a $l$ e la destra da $mid+1$ a $r$: $\Theta(n)$.
- **FindMaximumSubarray**$(a,l,r)$ — *Input*: un array e due indici $l\le r$. *Output*: gli indici $i\le j$ di un sottoarray di somma massima di $a[l..r]$ e la sua somma. Il migliore fra il sottoarray massimo della metà sinistra, della destra e di quelli che attraversano il punto medio (a parità, prima il sinistro, poi il destro). $S(n)=2S(n/2)+\Theta(n)=\Theta(n\log_2 n)$, $2n-1$ chiamate.

Il $-\infty$ dello pseudocodice è il valore più piccolo rappresentabile. Prima la traccia sull'esempio $a=[-2,5,-1,3]$ (come l'esecuzione dell'implementazione C++ del capitolo; risultato $(1,3,7)$, cioè $(2,4,7)$ con indici da 1), poi l'esempio della compravendita di azioni, con le variazioni giornaliere $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$: tabella dei risultati delle chiamate sui sottointervalli (per livello dell'albero, come nel capitolo; gli indici sono da 0) e i calcoli dei due `FindMaxCrossingSubarray` del livello 1. Il risultato è $(7,10,43)$, cioè $(8,11,43)$ con indici da 1.

In [18]:
%%writefile MaxSubDC.java
import java.util.*;

public class MaxSubDC {
    static int fmsCalls = 0;   // chiamate di FindMaximumSubarray

    static String tr(int[] t) {
        return "(" + t[0] + ", " + t[1] + ", " + t[2] + ")";
    }

    // Restituisce {maxleft, maxright, somma} del miglior sottoarray che attraversa mid
    static int[] findMaxCrossingSubarray(int[] a, int l, int mid, int r) {
        int leftsum = Integer.MIN_VALUE;        // -infinito
        int sum = 0;
        int maxleft = mid;
        for (int i = mid; i >= l; i--) {        // da mid a l
            sum = sum + a[i];                   // somma di a[i..mid]
            if (sum > leftsum) {
                leftsum = sum;
                maxleft = i;
            }
        }
        int rightsum = Integer.MIN_VALUE;
        sum = 0;
        int maxright = mid + 1;
        for (int j = mid + 1; j <= r; j++) {    // da mid+1 a r
            sum = sum + a[j];                   // somma di a[mid+1..j]
            if (sum > rightsum) {
                rightsum = sum;
                maxright = j;
            }
        }
        return new int[]{maxleft, maxright, leftsum + rightsum};
    }

    // Restituisce {i, j, somma} di un sottoarray di somma massima di a[l..r]
    static int[] findMaximumSubarray(int[] a, int l, int r) {
        fmsCalls++;
        if (l == r) {
            return new int[]{l, r, a[l]};       // caso base: un elemento
        }
        int mid = l + (r - l) / 2;
        int[] left = findMaximumSubarray(a, l, mid);
        int[] right = findMaximumSubarray(a, mid + 1, r);
        int[] cross = findMaxCrossingSubarray(a, l, mid, r);
        if (left[2] >= right[2] && left[2] >= cross[2]) {
            return left;
        }
        if (right[2] >= left[2] && right[2] >= cross[2]) {
            return right;
        }
        return cross;
    }

    // ---- varianti che stampano la traccia ----
    static int[] best(int[] left, int[] right, int[] cross) {
        if (left[2] >= right[2] && left[2] >= cross[2]) return left;
        if (right[2] >= left[2] && right[2] >= cross[2]) return right;
        return cross;
    }

    static int[] fmsTrace(int[] a, int l, int r, int depth) {
        String pad = " ".repeat(2 * depth);
        if (l == r) {
            System.out.printf("%sFindMaximumSubarray(a, %d, %d) = (%d, %d, %d)  caso base%n", pad, l, r, l, r, a[l]);
            return new int[]{l, r, a[l]};
        }
        int mid = l + (r - l) / 2;
        System.out.printf("%sFindMaximumSubarray(a, %d, %d): l != r, mid = %d%n", pad, l, r, mid);
        int[] left = fmsTrace(a, l, mid, depth + 1);
        int[] right = fmsTrace(a, mid + 1, r, depth + 1);
        int[] cross = findMaxCrossingSubarray(a, l, mid, r);
        System.out.printf("%s  FindMaxCrossingSubarray(a, %d, %d, %d) = %s%n", pad, l, mid, r, tr(cross));
        int[] ret = best(left, right, cross);
        System.out.printf("%s  left.sum=%d, right.sum=%d, cross.sum=%d => restituisce %s%n", pad, left[2], right[2], cross[2], tr(ret));
        return ret;
    }

    // ogni riga: {l, r, mid} + left + right + cross + ret
    static int[] fmsRows(int[] a, int l, int r, ArrayList<int[][]> rows) {
        if (l == r) {
            return new int[]{l, r, a[l]};
        }
        int mid = l + (r - l) / 2;
        int[] left = fmsRows(a, l, mid, rows);
        int[] right = fmsRows(a, mid + 1, r, rows);
        int[] cross = findMaxCrossingSubarray(a, l, mid, r);
        int[] ret = best(left, right, cross);
        rows.add(new int[][]{{l, r, mid}, left, right, cross, ret});
        return ret;
    }

    static void crossingTrace(int[] a, int l, int mid, int r) {
        System.out.printf("FindMaxCrossingSubarray(a, %d, %d, %d)%n", l, mid, r);
        System.out.printf("%3s %5s %5s %8s %8s%n", "i", "a[i]", "sum", "leftsum", "maxleft");
        int leftsum = Integer.MIN_VALUE, maxleft = mid, sum = 0;
        for (int i = mid; i >= l; i--) {
            sum = sum + a[i];
            if (sum > leftsum) {
                leftsum = sum;
                maxleft = i;
            }
            System.out.printf("%3d %5d %5d %8d %8d%n", i, a[i], sum, leftsum, maxleft);
        }
        System.out.printf("%3s %5s %5s %8s %8s%n", "j", "a[j]", "sum", "rightsum", "maxright");
        int rightsum = Integer.MIN_VALUE, maxright = mid + 1;
        sum = 0;
        for (int j = mid + 1; j <= r; j++) {
            sum = sum + a[j];
            if (sum > rightsum) {
                rightsum = sum;
                maxright = j;
            }
            System.out.printf("%3d %5d %5d %8d %8d%n", j, a[j], sum, rightsum, maxright);
        }
        System.out.printf("sottoarray che attraversa a[%d..%d], somma %d + %d = %d%n", maxleft, maxright, leftsum, rightsum, leftsum + rightsum);
    }

    public static void main(String[] args) {
        int[] a = {-2, 5, -1, 3};
        System.out.println("a = " + Arrays.toString(a));
        int[] res = fmsTrace(a, 0, 3, 0);
        fmsCalls = 0;
        findMaximumSubarray(a, 0, 3);
        System.out.printf("risultato %s   (%d, %d, %d con indici da 1)   chiamate=%d (2n-1=%d)%n", tr(res), res[0] + 1, res[1] + 1, res[2], fmsCalls, 2 * 4 - 1);

        System.out.println();
        int[] b = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
        int n = 16;
        System.out.println("a = " + Arrays.toString(b));
        fmsCalls = 0;
        res = findMaximumSubarray(b, 0, n - 1);
        System.out.printf("FindMaximumSubarray(a, 0, %d) = %s   (%d, %d, %d con indici da 1)   chiamate=%d (2n-1=%d)%n", n - 1, tr(res), res[0] + 1, res[1] + 1, res[2], fmsCalls, 2 * n - 1);
        ArrayList<int[][]> rows = new ArrayList<>();
        fmsRows(b, 0, n - 1, rows);
        rows.sort((x, y) -> (x[0][1] - x[0][0]) != (y[0][1] - y[0][0]) ? (y[0][1] - y[0][0]) - (x[0][1] - x[0][0]) : x[0][0] - y[0][0]);
        System.out.printf("%-9s %3s  %-13s %-13s %-13s restituito%n", "call", "mid", "left", "right", "crossing");
        for (int[][] w : rows) {
            String call = "[" + w[0][0] + ".." + w[0][1] + "]";
            System.out.printf("%-9s %3d  %-13s %-13s %-13s %s%n", call, w[0][2], tr(w[1]), tr(w[2]), tr(w[3]), tr(w[4]));
        }
        System.out.println();
        crossingTrace(b, 0, 3, 7);
        System.out.println();
        crossingTrace(b, 8, 11, 15);
    }
}

Writing MaxSubDC.java


In [19]:
!javac MaxSubDC.java && java MaxSubDC

a = [-2, 5, -1, 3]
FindMaximumSubarray(a, 0, 3): l != r, mid = 1
  FindMaximumSubarray(a, 0, 1): l != r, mid = 0
    FindMaximumSubarray(a, 0, 0) = (0, 0, -2)  caso base
    FindMaximumSubarray(a, 1, 1) = (1, 1, 5)  caso base
    FindMaxCrossingSubarray(a, 0, 0, 1) = (0, 1, 3)
    left.sum=-2, right.sum=5, cross.sum=3 => restituisce (1, 1, 5)
  FindMaximumSubarray(a, 2, 3): l != r, mid = 2
    FindMaximumSubarray(a, 2, 2) = (2, 2, -1)  caso base
    FindMaximumSubarray(a, 3, 3) = (3, 3, 3)  caso base
    FindMaxCrossingSubarray(a, 2, 2, 3) = (2, 3, 2)
    left.sum=-1, right.sum=3, cross.sum=2 => restituisce (3, 3, 3)
  FindMaxCrossingSubarray(a, 0, 1, 3) = (1, 3, 7)
  left.sum=5, right.sum=3, cross.sum=7 => restituisce (1, 3, 7)
risultato (1, 3, 7)   (2, 4, 7 con indici da 1)   chiamate=7 (2n-1=7)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
FindMaximumSubarray(a, 0, 15) = (7, 10, 43)   (8, 11, 43 con indici da 1)   chiamate=31 (2n-1=31)
call      mid  left 

**Prova tu.** Cambia l'array: il sottoarray massimo può avere un solo elemento o attraversare il punto medio; le chiamate sono sempre $2n-1$.

In [20]:
%%writefile ProvaMaxSubDC.java
import java.util.*;

public class ProvaMaxSubDC {
    public static void main(String[] args) {
        // Cambia l'array: quanti elementi ha il sottoarray massimo? Le chiamate sono sempre 2n-1.
        int[] b = {4, -1, 2, 1, -5, 4, -7, 3, 3};
        MaxSubDC.fmsCalls = 0;
        int[] t = MaxSubDC.findMaximumSubarray(b, 0, b.length - 1);
        System.out.println(MaxSubDC.tr(t) + " " + MaxSubDC.fmsCalls);
    }
}

Writing ProvaMaxSubDC.java


In [21]:
!javac ProvaMaxSubDC.java && java ProvaMaxSubDC

(0, 3, 6) 17


## 6. Kadane

**Kadane**$(a,n)$ — *Input*: un array $a[1..n]$ di $n$ interi. *Output*: gli indici $bl\le br$ di un sottoarray di somma massima e la somma $best$. Una sola scansione: $cur$ è la somma massima di un sottoarray che **termina** nella posizione corrente e $cl$ il suo primo indice; se $cur<0$ si riparte da $a[i]$, altrimenti si estende. $\Theta(n)$ passi, memoria extra $\Theta(1)$. (Nel capitolo 18 si trova la versione compatta che restituisce solo la somma.)

Sull'esempio della compravendita di azioni (variazioni giornaliere $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$) si stampa la tabella del capitolo: una riga per iterazione, con i valori alla fine dell'iterazione (indici da 0, la riga $i=0$ è l'inizializzazione). Il risultato $(7,10,43)$, cioè $(8,11,43)$ con indici da 1, coincide con quello dell'algoritmo divide et impera.

In [22]:
%%writefile Kadane.java
import java.util.*;

public class Kadane {
    // Restituisce {bl, br, best}: un sottoarray di somma massima di a[0..n-1]
    static int[] kadane(int[] a, int n) {
        int best = a[0];
        int bl = 0;
        int br = 0;
        int cur = a[0];             // somma massima di un sottoarray che termina in i
        int cl = 0;
        int i = 1;
        while (i < n) {
            if (cur < 0) {
                cur = a[i];         // riparte
                cl = i;
            } else {
                cur = cur + a[i];   // estende
            }
            if (cur > best) {
                best = cur;
                bl = cl;
                br = i;
            }
            i = i + 1;
        }
        return new int[]{bl, br, best};
    }

    // Come sopra, ma stampa una riga per iterazione
    static int[] kadaneTrace(int[] a, int n) {
        int best = a[0], bl = 0, br = 0;
        int cur = a[0], cl = 0;
        System.out.printf("%3d %5d  %-24s %4d %3d %5d  (%d, %d)%n", 0, a[0], "inizializzazione", cur, cl, best, bl, br);
        for (int i = 1; i < n; i++) {
            String action;
            if (cur < 0) {
                cur = a[i];
                cl = i;
                action = "riparte (cur<0)";
            } else {
                cur = cur + a[i];
                action = "estende";
            }
            if (cur > best) {
                best = cur;
                bl = cl;
                br = i;
            }
            System.out.printf("%3d %5d  %-24s %4d %3d %5d  (%d, %d)%n", i, a[i], action, cur, cl, best, bl, br);
        }
        return new int[]{bl, br, best};
    }

    public static void main(String[] args) {
        int[] a = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
        int n = 16;
        System.out.println("a = " + Arrays.toString(a));
        System.out.printf("%3s %5s  %-24s %4s %3s %5s  (bl, br)%n", "i", "a[i]", "azione", "cur", "cl", "best");
        int[] res = kadaneTrace(a, n);
        System.out.printf("risultato (%d, %d, %d)   (%d, %d, %d con indici da 1)%n", res[0], res[1], res[2], res[0] + 1, res[1] + 1, res[2]);
        int[] b = {-2, 5, -1, 3};
        int[] t = kadane(b, 4);
        System.out.printf("a = %s  Kadane(a, 4) = (%d, %d, %d)%n", Arrays.toString(b), t[0], t[1], t[2]);
    }
}

Writing Kadane.java


In [23]:
!javac Kadane.java && java Kadane

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
  i  a[i]  azione                    cur  cl  best  (bl, br)
  0    13  inizializzazione           13   0    13  (0, 0)
  1    -3  estende                    10   0    13  (0, 0)
  2   -25  estende                   -15   0    13  (0, 0)
  3    20  riparte (cur<0)            20   3    20  (3, 3)
  4    -3  estende                    17   3    20  (3, 3)
  5   -16  estende                     1   3    20  (3, 3)
  6   -23  estende                   -22   3    20  (3, 3)
  7    18  riparte (cur<0)            18   7    20  (3, 3)
  8    20  estende                    38   7    38  (7, 8)
  9    -7  estende                    31   7    38  (7, 8)
 10    12  estende                    43   7    43  (7, 10)
 11    -5  estende                    38   7    43  (7, 10)
 12   -22  estende                    16   7    43  (7, 10)
 13    15  estende                    31   7    43  (7, 10)
 14    -4  estende                   

**Prova tu.** Cambia l'array (per esempio con soli numeri negativi): Kadane riparte a ogni passo e trova il massimo elemento.

In [24]:
%%writefile ProvaKadane.java
import java.util.*;

public class ProvaKadane {
    public static void main(String[] args) {
        // Prova con un array di soli numeri negativi: Kadane riparte a ogni passo e trova il massimo elemento.
        int[] c = {-3, -1, -2, -5};
        int[] t = Kadane.kadane(c, c.length);
        System.out.println("(" + t[0] + ", " + t[1] + ", " + t[2] + ")");
    }
}

Writing ProvaKadane.java


In [25]:
!javac ProvaKadane.java && java ProvaKadane

(1, 1, -1)
